## Example 1: Basic TracerProvider setup

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry import trace

print("Example 1 - Basic TracerProvider setup")

# Create and configure tracer provider
provider = TracerProvider()
processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(processor)

# Set as global tracer provider
trace.set_tracer_provider(provider)

# Get tracer and create span
tracer = trace.get_tracer(__name__)
with tracer.start_as_current_span("example-span"):
    print("Span created successfully")

print("Example 1 completed successfully")

## Example 2: Resource configuration

In [ ]:
from opentelemetry.sdk.resources import Resource, SERVICE_NAME, SERVICE_VERSION
from opentelemetry.sdk.trace import TracerProvider

print("Example 2 - Resource configuration")

# Create resource with service attributes
resource = Resource.create({
    SERVICE_NAME: "my-service",
    SERVICE_VERSION: "1.0.0",
    "deployment.environment": "production"
})

print(f"Service name: {resource.attributes.get(SERVICE_NAME)}")
print(f"Service version: {resource.attributes.get(SERVICE_VERSION)}")

# Create provider with resource
provider = TracerProvider(resource=resource)
assert provider.resource.attributes[SERVICE_NAME] == "my-service"
assert provider.resource.attributes[SERVICE_VERSION] == "1.0.0"
print(f"TracerProvider created with resource")

print("Example 2 completed successfully")

## Example 3: BatchSpanProcessor

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry import trace

print("Example 3 - BatchSpanProcessor")

# Create batch processor with custom settings
exporter = ConsoleSpanExporter()
processor = BatchSpanProcessor(
    exporter,
    max_queue_size=2048,
    max_export_batch_size=512
)

provider = TracerProvider()
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)
with tracer.start_as_current_span("batch-span"):
    print("Batch processing configured")

print("Example 3 completed successfully")

## Example 4: MeterProvider setup

In [ ]:
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import ConsoleMetricExporter, PeriodicExportingMetricReader
from opentelemetry import metrics

print("Example 4 - MeterProvider setup")

# Create metric reader with console exporter
exporter = ConsoleMetricExporter()
reader = PeriodicExportingMetricReader(exporter, export_interval_millis=5000)

# Create and configure meter provider
provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)

# Get meter and create counter
meter = metrics.get_meter(__name__)
counter = meter.create_counter("requests", description="Request count")
counter.add(1, {"method": "GET"})

print("MeterProvider configured successfully")
print("Example 4 completed successfully")

## Example 5: Sampling configuration

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.sampling import TraceIdRatioBased, ParentBased
from opentelemetry import trace

print("Example 5 - Sampling configuration")

# Create sampling strategies
ratio_sampler = TraceIdRatioBased(0.5)  # Sample 50% of traces
parent_sampler = ParentBased(root=ratio_sampler)

# Create provider with sampler
provider = TracerProvider(sampler=parent_sampler)
trace.set_tracer_provider(provider)

print(f"Sampler configured: {type(parent_sampler).__name__}")
print(f"Root sampler: {type(ratio_sampler).__name__}")

# TraceIdRatioBased decides from the low 64 bits of the trace id, so these results are fixed
from opentelemetry.sdk.trace.sampling import Decision
assert ratio_sampler.should_sample(None, 1, "x").decision == Decision.RECORD_AND_SAMPLE
assert ratio_sampler.should_sample(None, 2**64 - 1, "x").decision == Decision.DROP

tracer = trace.get_tracer(__name__)
with tracer.start_as_current_span("sampled-span"):
    print("Span with sampling created")

print("Example 5 completed successfully")